# Coleta de Dados BCB

Notebook para coleta de dados historicos do Banco Central.

**Fontes:**
- **SGS**: Series temporais (Selic, CDI, PTAX, IBC-Br, IGP-M)
- **Expectations**: Expectativas do Relatorio Focus (IPCA, PIB, Cambio, Selic)

## 1. Setup

In [1]:
from pathlib import Path
from bacen.sgs import SGSCollector, SGS_CONFIG, list_indicators
from bacen.expectations import ExpectationsCollector, EXPECTATIONS_CONFIG, list_indicators as list_exp_indicators

# Caminho para dados
data_path = Path.cwd().parent / 'data'

print(f"Data path: {data_path}")
print(f"\nIndicadores SGS: {list_indicators()}")
print(f"\nIndicadores Expectations: {list_exp_indicators()}")

Data path: c:\Users\Enzo\Documents\Programming\dados-bcb\data

Indicadores SGS: ['ibc_br_bruto', 'ibc_br_dessaz', 'igp_m', 'selic', 'dolar_ptax', 'euro_ptax', 'cdi']

Indicadores Expectations: ['ipca_anual', 'igpm_anual', 'pib_anual', 'cambio_anual', 'selic_anual', 'ipca_mensal', 'igpm_mensal', 'cambio_mensal', 'selic', 'ipca_12m', 'ipca_24m', 'igpm_12m']


## 2. Coleta SGS

Coleta series temporais do SGS. Detecta automaticamente:
- **Primeira execucao**: Download historico completo
- **Execucoes seguintes**: Atualizacao incremental

In [2]:
# Inicializar coletor SGS
sgs_collector = SGSCollector(data_path)

# Coletar todos os indicadores
sgs_results = sgs_collector.collect()

ATUALIZACAO INCREMENTAL
Indicadores a coletar: 7

Atualizando IBC-Br Bruto desde 2025-10-01...
  Sem dados novos

Atualizando IBC-Br Dessazonalizado desde 2025-10-01...
  Sem dados novos

Atualizando IGP-M desde 2025-12-01...
  Sem dados novos

Meta Selic: Ja atualizado

Dolar PTAX: Ja atualizado

Euro PTAX: Ja atualizado

Atualizando CDI desde 2025-12-04...
  Sem dados novos

Coleta concluida!


In [3]:
# Status dos indicadores SGS
sgs_collector.get_status()

,arquivo,subdir,registros,colunas,primeira_data,ultima_data,status
0,cdi,sgs/daily,9946,1,1986-03-06,2025-12-03,OK
1,dolar_ptax,sgs/daily,10280,1,1984-11-28,2025-12-04,OK
2,euro_ptax,sgs/daily,6760,1,1998-12-31,2025-12-04,OK
3,selic,sgs/daily,9778,1,1999-03-05,2025-12-10,OK


## 3. Coleta Expectations

Coleta expectativas de mercado do Relatorio Focus.

In [4]:
# Inicializar coletor de expectativas
exp_collector = ExpectationsCollector(data_path)

# Coletar todas as expectativas
exp_results = exp_collector.collect()

COLETA DE EXPECTATIVAS DO BCB
Indicadores a coletar: 12

Coletando: top5_anuais [IPCA]...
  89,906 registros coletados
Salvo: raw\expectations\ipca_anual.parquet

Coletando: top5_anuais [IGP-M]...
  77,750 registros coletados
Salvo: raw\expectations\igpm_anual.parquet

Coletando: top5_anuais [PIB Total]...
  3,630 registros coletados
Salvo: raw\expectations\pib_anual.parquet

Coletando: top5_anuais [Cambio]...
  Nenhum dado retornado

Coletando: top5_anuais [Selic]...
  82,573 registros coletados
Salvo: raw\expectations\selic_anual.parquet

Coletando: mensais [IPCA]...
  173,832 registros coletados
Salvo: raw\expectations\ipca_mensal.parquet

Coletando: mensais [IGP-M]...
  122,194 registros coletados
Salvo: raw\expectations\igpm_mensal.parquet

Coletando: mensais [Cambio]...
  Nenhum dado retornado

Coletando: selic [Selic]...
  82,070 registros coletados
Salvo: raw\expectations\selic.parquet

Coletando: inflacao_12m [IPCA]...
  18,038 registros coletados
Salvo: raw\expectations\ipca_

In [5]:
# Status das expectativas
exp_collector.get_status()

,arquivo,subdir,registros,colunas,primeira_data,ultima_data,status
0,igpm_12m,expectations,14413,10,2001-11-07,2025-11-28,OK
1,igpm_anual,expectations,77750,9,2001-11-16,2025-11-28,OK
2,igpm_mensal,expectations,122194,10,2001-01-04,2025-11-28,OK
3,ipca_12m,expectations,18038,10,2001-11-07,2025-11-28,OK
4,ipca_24m,expectations,4696,10,2021-03-31,2025-11-28,OK
5,ipca_anual,expectations,89906,9,2001-11-06,2025-11-28,OK
6,ipca_mensal,expectations,173832,10,2000-01-03,2025-11-28,OK
7,pib_anual,expectations,3630,9,2023-03-31,2025-11-28,OK
8,selic,expectations,82070,10,2004-11-18,2025-11-28,OK
9,selic_anual,expectations,82573,9,2001-11-06,2025-11-28,OK


## 4. Consolidacao

Consolida dados em arquivos unificados para analise.

In [6]:
# Consolidar SGS (daily + monthly)
sgs_consolidated = sgs_collector.consolidate()

CONSOLIDANDO DADOS

Consolidando 4 arquivos de sgs/daily/...
Total: 13,337 registros, 4 colunas
  + Adicionada coluna 'cdi_anualizado'
  Salvo: processed\sgs_daily_consolidated.parquet
  sgs/daily: 13,337 registros x 5 colunas
  Periodo: 1984-11-28 a 2025-12-10

Consolidando 3 arquivos de sgs/monthly/...
Total: 437 registros, 3 colunas
  Salvo: processed\sgs_monthly_consolidated.parquet
  sgs/monthly: 437 registros x 3 colunas
  Periodo: 1989-07-01 a 2025-11-01

Consolidacao concluida!


In [7]:
# Consolidar Expectations
exp_consolidated = exp_collector.consolidate()

CONSOLIDANDO EXPECTATIVAS

Consolidando 10 arquivos de expectations/...
Total: 669,102 registros, 14 colunas
  Salvo: processed\expectations_consolidated.parquet
  expectations: 669,102 registros

Consolidacao concluida!
